<a href="https://colab.research.google.com/github/LikhithR916/BIS_Lab/blob/main/ParticleSwarmOptimization/1BM24CS147_WirelessSensorNetworkOptimization_ParticleSwarmOptimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# ---------------- WSN PARAMETERS ----------------

N = 30
AREA = 100
BS = np.array([50, 150])

E_INIT = 0.5
E_ELEC = 50e-9
E_FS = 10e-12
E_MP = 0.0013e-12
E_DA = 5e-9

PACKET = 4000
K = 4

# ---------------- PSO PARAMETERS ----------------

PARTICLES = 20
ITERATIONS = 50
W = 0.7
C1 = 1.5
C2 = 1.5

# ---------------- INITIALIZE WSN ----------------

nodes = np.random.rand(N, 2) * AREA
energy = np.ones(N) * E_INIT

# ---------------- ENERGY MODEL ----------------

def transmission_energy(distance):
    d0 = np.sqrt(E_FS / E_MP)

    if distance < d0:
        return PACKET * E_ELEC + PACKET * E_FS * distance**2
    else:
        return PACKET * E_ELEC + PACKET * E_MP * distance**4


def reception_energy():
    return PACKET * E_ELEC


# ---------------- FITNESS FUNCTION ----------------

def fitness(position):
    ch = np.argsort(position)[-K:]
    energy_used = np.zeros(N)

    for i in range(N):

        if i in ch:
            members = []

            for j in range(N):
                if j not in ch:
                    nearest_ch = ch[
                        np.argmin(np.linalg.norm(nodes[ch] - nodes[j], axis=1))
                    ]

                    if nearest_ch == i:
                        members.append(j)

            for j in members:
                energy_used[i] += reception_energy()
                energy_used[i] += E_DA * PACKET

            distance = np.linalg.norm(nodes[i] - BS)
            energy_used[i] += transmission_energy(distance)

        else:
            nearest_ch = ch[
                np.argmin(np.linalg.norm(nodes[ch] - nodes[i], axis=1))
            ]

            distance = np.linalg.norm(nodes[i] - nodes[nearest_ch])
            energy_used[i] += transmission_energy(distance)

    return np.sum(energy_used) + 0.2 * np.max(energy_used)


# ---------------- INITIALIZE PSO ----------------

X = np.random.rand(PARTICLES, N)
V = np.random.uniform(-1, 1, (PARTICLES, N))

Pbest = X.copy()
Pbest_fitness = np.array([fitness(x) for x in X])

best_index = np.argmin(Pbest_fitness)
Gbest = Pbest[best_index].copy()

# ---------------- PSO ITERATIONS ----------------

for iteration in range(ITERATIONS):

    for i in range(PARTICLES):

        r1 = np.random.rand(N)
        r2 = np.random.rand(N)

        V[i] = (
            W * V[i]
            + C1 * r1 * (Pbest[i] - X[i])
            + C2 * r2 * (Gbest - X[i])
        )

        X[i] = np.clip(X[i] + V[i], 0, 1)

        current_fitness = fitness(X[i])

        if current_fitness < Pbest_fitness[i]:
            Pbest[i] = X[i].copy()
            Pbest_fitness[i] = current_fitness

        if current_fitness < fitness(Gbest):
            Gbest = X[i].copy()

# ---------------- FINAL RESULTS ----------------

cluster_heads = np.argsort(Gbest)[-K:]

print("\n--- WSN Optimization Results ---")
print("Optimal Cluster Heads:", cluster_heads)
print("Minimum Energy Consumption:", round(fitness(Gbest), 6), "J")
print("Alive Sensor Nodes:", np.sum(energy > 0))
print("Total Residual Energy:", round(np.sum(energy), 4), "J")


--- WSN Optimization Results ---
Optimal Cluster Heads: [ 8 17 20 16]
Minimum Energy Consumption: 0.014656 J
Alive Sensor Nodes: 30
Total Residual Energy: 15.0 J
